# WP15 — Multi-Objective Safety Aggregation
**Prometheus v0.99**

Unifies the independent safety verdicts from WP2–WP14 into a single, auditable composite decision via three-layer architecture:

1. **Normalisation** — Each WP verdict is mapped to `(safety_score ∈ [0,1], severity ∈ [0,10])` via a thin `VerdictAdapter`
2. **Aggregation** — Three strategies: `WeightedSumAggregator`, `ParetoFrontAggregator`, `ConformalPValueAggregator` (Fisher 1932)
3. **Gate** — `CompositeSafetyGate` exposes a unified interface compatible with `CorrigibilityGate` / `MCSSupervisor`

**References**: Fisher (1932) *Statistical Methods for Research Workers*; Pareto (1906); Soares et al. (2015) *Corrigibility*

In [ ]:
import sys, os
if 'google.colab' in sys.modules:
    os.system('git clone https://github.com/prometheus-ai/Prometheus_v0_PoC /content/Prometheus_v0_PoC 2>/dev/null || true')
    os.system('pip install scipy -q')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')
print('Environment ready.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from prometheus.multi_objective_safety import (
    NormalisedVerdict, CompositeSafetyVerdict, SafetyLevel,
    WeightedSumAggregator, ParetoFrontAggregator, ConformalPValueAggregator,
    CompositeSafetyGate, make_composite_gate, composite_safety_summary_table,
    _BUILTIN_ADAPTERS,
)
from benchmarks.multi_objective_safety_benchmark import (
    MultiObjectiveSafetyBenchmark,
    _safe_ood, _unsafe_ood, _safe_fv, _unsafe_fv,
    _safe_debate, _unsafe_debate, _safe_hacking, _unsafe_hacking,
    _safe_corr, _unsafe_corr,
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
rng = np.random.default_rng(42)
print('Imports ready. Adapters:', list(_BUILTIN_ADAPTERS.keys()))

---
## 1 — Verdict Adapters: Normalisation

In [ ]:
# Show safe vs unsafe normalised scores for each adapter
adapter_pairs = {
    'WP2 OOD'       : (_BUILTIN_ADAPTERS['WP2'],  _safe_ood(),     _unsafe_ood()),
    'WP7 FV'        : (_BUILTIN_ADAPTERS['WP7'],  _safe_fv(),      _unsafe_fv(9)),
    'WP10 Debate'   : (_BUILTIN_ADAPTERS['WP10'], _safe_debate(),  _unsafe_debate()),
    'WP12 Hacking'  : (_BUILTIN_ADAPTERS['WP12'], _safe_hacking(), _unsafe_hacking()),
    'WP14 Corr'     : (_BUILTIN_ADAPTERS['WP14'], _safe_corr(),    _unsafe_corr(9)),
}

labels      = list(adapter_pairs.keys())
safe_scores = [adapter_pairs[k][0].adapt(adapter_pairs[k][1]).safety_score for k in labels]
unsafe_scores = [adapter_pairs[k][0].adapt(adapter_pairs[k][2]).safety_score for k in labels]

x  = np.arange(len(labels))
w  = 0.35
fig, ax = plt.subplots(figsize=(11, 4))
b1 = ax.bar(x - w/2, safe_scores,   w, label='Safe verdict',   color='#2ecc71', alpha=0.85, edgecolor='white')
b2 = ax.bar(x + w/2, unsafe_scores, w, label='Unsafe verdict', color='#e74c3c', alpha=0.85, edgecolor='white')
ax.axhline(0.7, color='#2ecc71', ls='--', lw=1, label='Safe threshold ≥0.7')
ax.axhline(0.4, color='#e74c3c', ls='--', lw=1, label='Unsafe threshold ≤0.4')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylabel('Normalised safety_score')
ax.set_title('Adapter Normalisation: Safe vs Unsafe Raw Verdicts', fontweight='bold')
ax.set_ylim(0, 1.25); ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

print(f'\n{"Adapter":<16} {"Safe score":>12} {"Unsafe score":>14}')
print('-' * 44)
for k, s, u in zip(labels, safe_scores, unsafe_scores):
    print(f'{k:<16} {s:>12.3f} {u:>14.3f}')

---
## 2 — Aggregation Strategies Compared

In [ ]:
# Sweep composite score as we replace safe WPs one-by-one with unsafe ones
wp_ids = ['WP2', 'WP7', 'WP10', 'WP12', 'WP14']

conformal = ConformalPValueAggregator(alpha=0.05)
for wp in wp_ids:
    conformal.calibrate(wp, [0.8, 0.85, 0.9, 0.95, 1.0])

aggregators = [
    ('Weighted Sum',   WeightedSumAggregator()),
    ('Pareto Front',   ParetoFrontAggregator()),
    ('Conformal p',    conformal),
]

n_unsafe_range = range(6)   # 0 to 5 unsafe WPs
scores = {name: [] for name, _ in aggregators}

for n_unsafe in n_unsafe_range:
    verts = []
    for i, wp in enumerate(wp_ids):
        if i < n_unsafe:
            verts.append(NormalisedVerdict(wp, wp, 0.0, 0.9, 9.0))
        else:
            verts.append(NormalisedVerdict(wp, wp, 1.0, 0.9, 0.0))
    for name, agg in aggregators:
        s, _, _ = agg.aggregate(verts, threshold=0.5)
        scores[name].append(s)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#3498db', '#e67e22', '#9b59b6']
for (name, _), color in zip(aggregators, colors):
    ax.plot(list(n_unsafe_range), scores[name], 'o-', label=name, color=color, lw=2)
ax.axhline(0.5, color='gray', ls='--', lw=1, label='Decision threshold 0.5')
ax.set_xlabel('Number of unsafe WP verdicts (out of 5)')
ax.set_ylabel('Composite safety score')
ax.set_title('Aggregation Strategies: Composite Score vs Number of Unsafe WPs', fontweight='bold')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

---
## 3 — Veto Propagation

In [ ]:
gate = make_composite_gate(method='weighted_sum', veto_severity=8.0)

cases = {
    'All safe'             : {'WP7': _safe_fv(),       'WP14': _safe_corr(),      'WP12': _safe_hacking()},
    'One veto (sev=9)'     : {'WP7': _safe_fv(),       'WP14': _unsafe_corr(9),   'WP12': _safe_hacking()},
    'One mild unsafe (sev=4)': {'WP7': _safe_fv(),     'WP14': _unsafe_corr(4),   'WP12': _safe_hacking()},
    'All unsafe'           : {'WP7': _unsafe_fv(9),    'WP14': _unsafe_corr(9),   'WP12': _unsafe_hacking()},
    'All abstain'          : {'WP7': None,              'WP14': None,              'WP12': None},
}

verdicts = {name: gate.evaluate(raw) for name, raw in cases.items()}

print(f'\n{"Case":<28} {"Allowed":>8} {"Score":>7} {"MaxSev":>8} {"Vetoed by"}')
print('-' * 68)
for name, v in verdicts.items():
    sym   = 'YES' if v.allowed else 'NO'
    vetos = ','.join(v.vetoed_by) if v.vetoed_by else '-'
    print(f'{name:<28} {sym:>8} {v.composite_score:>7.3f} {v.max_severity:>8.1f} {vetos}')

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
names  = list(verdicts.keys())
cscores = [verdicts[n].composite_score for n in names]
colors = ['#2ecc71' if verdicts[n].allowed else '#e74c3c' for n in names]
ax.barh(names, cscores, color=colors, edgecolor='white', alpha=0.85)
ax.axvline(0.5, color='gray', ls='--', lw=1.5, label='Threshold 0.5')
ax.set_xlabel('Composite safety score')
ax.set_title('Composite Score per Case', fontweight='bold')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax2 = axes[1]
maxsevs = [verdicts[n].max_severity for n in names]
ax2.barh(names, maxsevs, color=['#e74c3c' if s >= 8 else '#f39c12' if s > 0 else '#2ecc71'
                                  for s in maxsevs], edgecolor='white', alpha=0.85)
ax2.axvline(8.0, color='navy', ls='--', lw=1.5, label='Veto threshold 8')
ax2.set_xlabel('Max severity')
ax2.set_title('Max Severity per Case', fontweight='bold')
ax2.legend(fontsize=9)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout(); plt.show()

---
## 4 — Pareto Front Visualisation

In [ ]:
# Generate random verdict sets and plot in (score, certainty) space
gate_pareto = make_composite_gate('pareto_front', score_threshold=0.5, cert_threshold=0.4)

points = []
for _ in range(200):
    # Random mixture of safe/unsafe WPs
    safe_mask = rng.random(5) > rng.uniform(0.2, 0.8)
    verts = {}
    for i, (wp, safe) in enumerate(zip(['WP2','WP7','WP10','WP12','WP14'], safe_mask)):
        if safe:
            verts[wp] = [_safe_ood, _safe_fv, _safe_debate, _safe_hacking, _safe_corr][i]()
        else:
            verts[wp] = [_unsafe_ood, lambda: _unsafe_fv(6), _unsafe_debate,
                         _unsafe_hacking, lambda: _unsafe_corr(6)][i]()
    v = gate_pareto.evaluate(verts)
    points.append((v.composite_score, v.composite_certainty, v.allowed))

fig, ax = plt.subplots(figsize=(8, 6))
allowed_pts  = [(s, c) for s, c, a in points if a]
blocked_pts  = [(s, c) for s, c, a in points if not a]

if allowed_pts:
    ax.scatter(*zip(*allowed_pts), color='#2ecc71', alpha=0.6, s=40, label='Allowed', zorder=3)
if blocked_pts:
    ax.scatter(*zip(*blocked_pts), color='#e74c3c', alpha=0.6, s=40, label='Blocked', marker='x', zorder=3)

# Reference threshold rectangle
ax.axvline(0.5, color='navy', ls='--', lw=1.5, label='Score threshold 0.5')
ax.axhline(0.4, color='gray', ls='--', lw=1.5, label='Certainty threshold 0.4')
ax.set_xlabel('Composite safety score')
ax.set_ylabel('Composite certainty')
ax.set_title('Pareto Front: (Score, Certainty) Decision Space', fontweight='bold')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()
print(f'Allowed: {len(allowed_pts)}/200, Blocked: {len(blocked_pts)}/200')

---
## 5 — Full Benchmark: 4 Scenarios

In [ ]:
bench   = MultiObjectiveSafetyBenchmark(seed=42)
results = bench.run_all()
print(bench.summary_table(results))

In [ ]:
an_r  = next(r for r in results if r.scenario == 'adapter_normalisation')
ac_r  = next(r for r in results if r.scenario == 'aggregator_consistency')
vp_r  = next(r for r in results if r.scenario == 'veto_propagation')
e2_r  = next(r for r in results if r.scenario == 'end_to_end_gate')

fig, axes = plt.subplots(1, 4, figsize=(17, 4))

# (A) Adapter normalisation
ax = axes[0]
n  = an_r.metrics['n_adapters']
vals = [an_r.metrics['safe_ok']/n, an_r.metrics['unsafe_ok']/n,
        an_r.metrics['abstain_ok']/n]
bars = ax.bar(['Safe\ncorrect', 'Unsafe\ncorrect', 'Abstain\ncorrect'],
              vals, color=['#2ecc71','#e74c3c','#95a5a6'], edgecolor='white', alpha=0.85)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.0%}', ha='center', fontweight='bold')
ax.set_ylim(0, 1.2); ax.set_title('(A) Adapter\nNormalisation', fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# (B) Aggregator consistency
ax2 = axes[1]
bars2 = ax2.bar(['Safe agreed', 'Unsafe agreed'],
                [ac_r.metrics['safe_agreed']/3, ac_r.metrics['unsafe_agreed']/3],
                color=['#2ecc71','#e74c3c'], edgecolor='white', alpha=0.85)
for bar, v in zip(bars2, [ac_r.metrics['safe_agreed']/3, ac_r.metrics['unsafe_agreed']/3]):
    ax2.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.0%}', ha='center', fontweight='bold')
ax2.set_ylim(0, 1.2); ax2.set_title('(B) Aggregator\nConsistency (3/3)', fontweight='bold')
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

# (C) Veto propagation scores
ax3 = axes[2]
vp_scores = [
    vp_r.metrics['all_safe_score'],
    vp_r.metrics['low_sev_score'],
]
vp_labels = ['All-safe\nscore', 'Low-sev\nunsafe score']
colors3 = ['#2ecc71', '#f39c12']
b3 = ax3.bar(vp_labels, vp_scores, color=colors3, edgecolor='white', alpha=0.85)
for bar, v in zip(b3, vp_scores):
    ax3.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.3f}', ha='center', fontweight='bold')
ax3.set_ylim(0, 1.2)
veto_blocked = not vp_r.metrics['veto_allowed']
ax3.set_title(f'(C) Veto Propagation\nveto_blocked={veto_blocked}', fontweight='bold')
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)

# (D) End-to-end gate
ax4 = axes[3]
bd = e2_r.metrics['block_rate_dangerous']
bc = e2_r.metrics['block_rate_clean']
bars4 = ax4.bar(['Block dangerous', 'Block clean'],
                [bd, bc], color=['#e74c3c','#2ecc71'], edgecolor='white', alpha=0.85)
for bar, v in zip(bars4, [bd, bc]):
    ax4.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.0%}', ha='center', fontweight='bold')
ax4.axhline(0.80, color='#e74c3c', ls='--', lw=1, label='Danger target ≥80%')
ax4.axhline(0.10, color='#2ecc71', ls='--', lw=1, label='Clean target ≤10%')
ax4.set_ylim(0, 1.2); ax4.legend(fontsize=8)
ax4.set_title('(D) End-to-End Gate', fontweight='bold')
ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)

n_pass = sum(r.passed for r in results)
fig.suptitle(f'WP15 Multi-Objective Safety Benchmark — Prometheus v0.99  [{n_pass}/{len(results)} passed]',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()

---
## Summary

| Property | Mechanism | Result |
|----------|-----------|--------|
| Adapter normalisation | 9 thin adapters, safe→≥0.7, unsafe→≤0.4 | **100% correct** |
| Aggregator agreement | WeightedSum / Pareto / Conformal on extreme cases | **3/3 agree** |
| Veto propagation | Single sev≥8 WP blocks regardless of others | **correct** |
| End-to-end block rate | Dangerous ≥80%, clean ≤10% | **passes** |

**Test coverage**: 79 tests, all passing (`pytest tests/test_multi_objective_safety.py -v`)

**Key design decisions**:
- Adapters are read-only wrappers — no WP module is modified
- `veto` flag (severity ≥ veto_severity) overrides the aggregator decision
- `abstain` verdicts are excluded from aggregation (not penalised)
- `ConformalPValueAggregator` requires a calibration corpus of known-safe scores per WP
- `ParetoFrontAggregator` is the most conservative: requires dominance on *both* score and certainty

**Files**:
- `prometheus/multi_objective_safety.py` — adapters, aggregators, CompositeSafetyGate
- `benchmarks/multi_objective_safety_benchmark.py` — 4 scenarios
- `tests/test_multi_objective_safety.py` — 79-test suite
- `notebooks/wp15_multi_objective_safety_demo.ipynb` — this notebook